In [ ]:
# %%
import matplotlib.pyplot as plt
import gc
import os
import shutil
import sys
import time
import warnings
from functools import partial
import torch.distributed as tdist
import torch
from PIL import Image
from torch.utils.data import DataLoader
from utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates
import numpy as np
import dist
from utils import arg_util, misc
from utils.data import build_dataset
from utils.data_sampler import DistInfiniteBatchSampler, EvalDistributedSampler
from utils.misc import auto_resume
import math
from models import SRVAR, VQVAE, build_vae_srvar

args: arg_util.Args = arg_util.Args()
# args.data = "./data/df2k_ost/GT_resized"
# args.data = "./data/DIV2K_train_HR"
# args.data = "./data/train"
# args.data = "./data/brats_256_t1_new/train"
args.batch_size = 1
args.fp16=1
args.alng = 1e-3
args.wpe = 0.1
args.pn = "1M"
args.rope2d_normalized_by_hw = 2
args.rope2d_each_sa_layer = 1
args.enable_checkpointing = "full-block"
args.tlen = 1024
args.device = "cuda"

args.Ct5 = 32
args.vocab_size = 4096
maxtot = 5
args.use_diff = False
args.use_ref = False
beam_search_nums = 0
choose_min = "max"
score_compare = "max"


args.seed = 666
args.seed_everything(False)

In [ ]:
ckpt_path_srvar = f"local_output/ar-ckpt-last.pth"
V=args.vocab_size
Cvae=args.Ct5
ch=160
share_quant_resi=4
# patch_nums=(1, 3, 5 , 8, 12, 16)   # 10 steps by default
patch_nums=(1, 2, 3, 4, 5, 6, 8, 10, 13, 16) 

# print(torch.load(ckpt_path, map_location='cpu')['trainer'].keys())
vae = VQVAE(vocab_size=V, z_channels=Cvae, ch=ch, test_mode=True, share_quant_resi=share_quant_resi, v_patch_nums=patch_nums).to(args.device)
    
srvar_kw = dict(
    low_channel=args.Ct5, low_len=args.tlen,
    norm_eps=args.norm_eps, rms_norm=args.rms,
    shared_aln=args.saln, head_aln=args.haln,
    cond_drop_rate=args.cfg, rand_uncond=args.rand_uncond, drop_rate=args.drop,
    cross_attn_layer_scale=args.ca_gamma, nm0=args.nm0, tau=args.tau, cos_attn=args.cos, swiglu=args.swi,
    raw_scale_schedule=patch_nums,
    head_depth=args.dec,
    top_p=args.tp, top_k=args.tk,
    customized_flash_attn=args.flash, fused_mlp=args.fuse, fused_norm=args.fused_norm,
    checkpointing=args.enable_checkpointing,
    pad_to_multiplier=args.pad_to_multiplier,
    use_flex_attn=args.use_flex_attn,
    batch_size=args.batch_size,
    add_lvl_embeding_only_first_block=args.add_lvl_embeding_only_first_block,
    rope2d_each_sa_layer=args.rope2d_each_sa_layer,
    rope2d_normalized_by_hw=args.rope2d_normalized_by_hw,
    pn=args.pn,
    train_h_div_w_list=None,
    always_training_scales=args.always_training_scales,
    apply_spatial_patchify=args.apply_spatial_patchify,
    block_chunks = args.block_chunks,
    use_diff = args.use_diff,
    use_ref = args.use_ref,

)
if args.dp >= 0: srvar_kw['drop_path_rate'] = args.dp
if args.hd > 0: srvar_kw['num_heads'] = args.hd

# print(f'[create srvar] constructor kw={srvar_kw}\n')

# vae.load_state_dict(torch.load("vae_ch160v4096z32.pth", map_location='cpu'))
vae.load_state_dict(torch.load(ckpt_path_srvar, map_location='cpu')['trainer']['vae_local'])
srvar_kw['vae_local'] = vae


srvar: SRVAR = SRVAR(**srvar_kw)
srvar = srvar.to(args.device)
srvar.load_state_dict(torch.load(ckpt_path_srvar, map_location='cpu')['trainer']['srvar_wo_ddp'])

srvar.eval()

vae.eval()


In [ ]:
args.data_path = "../vaex/data/mix_data/"
dataset_train, dataset_val = build_dataset(
    args.data_path,augment=False,use_ref=False
)
types = str((type(dataset_train).__name__, type(dataset_val).__name__))

ld_val = DataLoader(
    dataset_val, num_workers=args.workers, batch_size=args.batch_size,shuffle=False,
)


ld_train = DataLoader(
    dataset=dataset_train, num_workers=args.workers,batch_size=args.batch_size,shuffle=True,
)
del dataset_val,dataset_train

In [ ]:
def setup(rank, world_size):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12115'
    tdist.init_process_group("nccl", rank=rank, world_size=world_size)
setup(0,1)

In [ ]:
for idx, datas  in enumerate(ld_val):
    if args.use_ref:
        inp_B3HW_low, inp_B3HW_super, ref_B3HW = datas
    else :
        inp_B3HW_low, inp_B3HW_super = datas
        ref_B3HW = None
    
    inp_B3HW_low = inp_B3HW_low.to(dist.get_device(), non_blocking=True)
    inp_B3HW_super = inp_B3HW_super.to(dist.get_device(), non_blocking=True)
    ref_B3HW = ref_B3HW.to(dist.get_device(), non_blocking=True) if ref_B3HW is not None else None
    
    B, V = inp_B3HW_low.shape[0], vae.vocab_size
    
    gt_idx_Bl_super= vae.img_to_idxBl(inp_B3HW_super)
    gt_BL_super = torch.cat(gt_idx_Bl_super, dim=1)
    x_BLCv_wo_first_l_super= vae.quantize.idxBl_to_var_input(gt_idx_Bl_super)
    
    h_div_w = inp_B3HW_low.shape[-2] / inp_B3HW_low.shape[-1]
    T = 1 if inp_B3HW_low.dim() == 4 else inp_B3HW_low.shape[2]
    h_div_w_templates = np.array(list(dynamic_resolution_h_w.keys()))
    h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w-h_div_w_templates))]
    scale_schedule = dynamic_resolution_h_w[h_div_w_template]["1M"]['scales']
    scale_schedule = [ (min(t, T//4+1), h, w) for (t,h, w) in scale_schedule]
    
    ret, idx_Bl_list, img = srvar.autoregressive_infer_cfg(vae=vae, inp_B3HW_low=inp_B3HW_low, ref_B3HW = ref_B3HW, 
                        scale_schedule=scale_schedule,
                        ret_img=True,
                        B=B,
                        choose_min=choose_min,
                        score_compare=score_compare,
                        beam_search_nums = beam_search_nums
                        )
    
    nup_test = vae.idxBl_to_img(idx_Bl_list, same_shape=True, last_one=False)
    nup_gt = vae.idxBl_to_img(gt_idx_Bl_super, same_shape=True, last_one=False)
    
    def process_image(x):
        """处理单张图片"""    
        x = x[0].detach().cpu().permute(1, 2, 0).numpy()  # 转换为 HWC 格式的 numpy 数组
        x = (x * 0.5 + 0.5) * 255  # 反归一化并缩放到 [0, 255]
        x = x.astype(np.uint8)  # 转换为 uint8
        return x

    def concatenate_images(images, axis):
        """沿指定轴拼接图片"""
        return np.concatenate(images, axis=axis)
    
    test_combined = concatenate_images([process_image(x) for x in nup_test], axis=1)   # 左右连接
    gt_combined = concatenate_images([process_image(x) for x in nup_gt], axis=1)      # 左右连接
    
    inp_B3HW_low = process_image(inp_B3HW_low)
    inp_B3HW_super = process_image(inp_B3HW_super)
    
    test_combined = concatenate_images([test_combined,inp_B3HW_super, inp_B3HW_low],axis=1)
    gt_combined = concatenate_images([gt_combined,inp_B3HW_super, inp_B3HW_low],axis=1)
    
    # 将三个结果上下连接
    final_image = concatenate_images([test_combined,  gt_combined], axis=0)  # 上下连接
    # Image.fromarray(final_image)
    
    plt.imshow( concatenate_images([img[0].cpu().numpy(),inp_B3HW_super,process_image(nup_gt[-1])],axis=1))
    # 生成图片(可能diffusion后) 真实图片 vae重建图片
    plt.axis('off')  # 可选，关闭坐标轴
    plt.show()
    plt.imshow(final_image)
    # 图片生成pipeline 真实图片 LR图片
    plt.axis('off')  # 可选，关闭坐标轴
    plt.show()
    if idx > maxtot:
        break